# Madrid por distritos — análisis REAL (INE 2023) + mapas

Análisis completo con datos **oficiales** del INE (Atlas de Renta, ADRH 2023) a nivel de los **21 distritos** de Madrid, más el **mapa real** de la ciudad.

## Flujo de datos (cómo entra la información)
```txt
Fuentes oficiales         →  extractor          →  data/raw     →  loader         →  análisis
INE Atlas (4 tablas)         python src/extract.py    (crudo)        src/load_ine.py    este notebook
Geometría (Ayto. Madrid)                                            src/maps.py
```

**Requisito (una vez):** descargar los datos con  `python src/extract.py`  (deja 4 CSV del INE + `distritos_madrid.zip` en `data/raw/`). Ver `docs/flujo` y `00_documentacion/flujo_anadir_fuente_datos.md`.

> Para añadir más fuentes (alquiler, compra...): se siguen los mismos pasos — URL en `extract.py`, loader en `src/`, y se cruza aquí por distrito.


## 0. Configuración (ejecuta esto primero)


In [ ]:
import sys, os, importlib
SRC = os.path.abspath('../src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import matplotlib.pyplot as plt
import config, load_ine, maps
importlib.reload(load_ine); importlib.reload(maps)

# comprobar que los datos están descargados
ine = ['ine_renta_media_mediana.csv','ine_fuente_ingresos.csv','ine_gini_p80p20.csv','ine_demografia.csv']
faltan = [f for f in ine + ['distritos_madrid.zip'] if not (config.DATA_RAW / f).exists()]
if faltan:
    print('⚠️  Faltan ficheros en data/raw:', faltan)
    print('   Ejecuta en la terminal:  python src/extract.py')
else:
    print('✅ Datos listos en data/raw')


## 1. Carga de datos reales (4 fuentes del INE unidas por distrito)

`load_ine.construir_madrid()` lee los 4 CSV, los limpia, filtra los 21 distritos de Madrid y los une en una sola tabla.


In [ ]:
mad = load_ine.construir_madrid()
print(mad.shape[0], 'distritos ·', mad.shape[1], 'columnas')
mad[['cod_distrito','nombre_distrito','Renta neta media por hogar',
     'Mediana de la renta por unidad de consumo','Índice de Gini']].head(21)


## 2. Ranking de renta por distrito

Renta neta media por hogar (€/año). Rojo = por debajo de la mediana; azul = por encima.


In [ ]:
col = 'Renta neta media por hogar'
d = mad.dropna(subset=[col]).sort_values(col)
colors = ['#de2d26' if v < d[col].median() else '#2c7fb8' for v in d[col]]
fig, ax = plt.subplots(figsize=(9,8))
ax.barh(d['nombre_distrito'], d[col], color=colors)
ax.set_xlabel(col + ' (€/año)')
ax.set_title('Madrid · renta REAL por distrito (INE 2023)')
plt.tight_layout(); plt.show()

print('Más rico: ', d.iloc[-1]['nombre_distrito'], int(d.iloc[-1][col]), '€')
print('Más pobre:', d.iloc[0]['nombre_distrito'], int(d.iloc[0][col]), '€')
print('Ratio rico/pobre:', round(d.iloc[-1][col]/d.iloc[0][col], 2), 'veces')


## 3. Renta vs desigualdad (¿los distritos ricos son más desiguales?)

Cruce renta × **Índice de Gini** (desigualdad dentro del distrito). Tamaño = población; color = % de renta que viene de salario.


In [ ]:
x, y = 'Renta neta media por hogar', 'Índice de Gini'
color = 'ingresos_Fuente de ingreso: salario'
size = 'demo_Población'
fig, ax = plt.subplots(figsize=(8.5,6.5))
sc = ax.scatter(mad[x], mad[y], s=mad[size]/2500, c=mad[color], cmap='RdYlGn')
for _, r in mad.iterrows():
    ax.annotate(r['nombre_distrito'], (r[x], r[y]), fontsize=6, alpha=0.75)
ax.set_xlabel(x + ' (€)'); ax.set_ylabel(y + ' (desigualdad)')
ax.set_title('Madrid · renta vs desigualdad (INE 2023)')
plt.colorbar(sc, label='% de renta que viene de salario')
plt.tight_layout(); plt.show()

print('Correlación renta ↔ Gini:', round(mad[x].corr(mad[y]), 2))
print('Correlación renta ↔ %salario:', round(mad[x].corr(mad[color]), 2))


## 4. Capa DuckDB — consultar los datos con SQL

Guardamos la tabla en **DuckDB** (una base de datos en un fichero, sin servidor) y la consultamos con SQL. Las consultas devuelven DataFrames de pandas.


In [ ]:
import duckdb
config.OUTPUTS.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(config.OUTPUTS / 'madrid.duckdb'))
con.execute('CREATE OR REPLACE TABLE distritos AS SELECT * FROM mad')

top = con.execute('''
    SELECT nombre_distrito,
           "Renta neta media por hogar" AS renta_hogar,
           "Índice de Gini"            AS gini
    FROM distritos ORDER BY renta_hogar DESC LIMIT 5
''').df()
con.close()
top


## 5. Mapa REAL de Madrid (coroplético)

Unimos la geometría real de los distritos (shapefile del Ayto. de Madrid, `data/raw/distritos_madrid.zip`) con la renta y la pintamos.


In [ ]:
gdf = maps.unir_distritos_madrid(str(config.DATA_RAW / 'distritos_madrid.zip'), mad)
fig, ax = maps.mapa_choropleth(
    gdf, 'Renta neta media por hogar',
    'Madrid · renta neta media por hogar por distrito (INE 2023)',
    cmap='viridis')
plt.show()


## 6. Mapa interactivo (Kepler.gl)

Genera un mapa interactivo y lo guarda como HTML autónomo en `outputs/`. Si el widget no se ve en Jupyter, abre el `.html` en el navegador (siempre funciona).


In [ ]:
cols = ['nombre_distrito', 'Renta neta media por hogar', 'Índice de Gini', 'geometry']
try:
    m = maps.mapa_kepler(gdf[cols], 'distritos',
                         guardar_html=str(config.OUTPUTS / 'mapa_madrid_renta.html'))
    print('Mapa guardado en outputs/mapa_madrid_renta.html')
    display(m)
except Exception as e:
    print('Kepler no se pudo mostrar en el notebook:', e)
    print('Abre el HTML en el navegador: outputs/mapa_madrid_renta.html')


## 7. Conclusiones y próximos pasos

**Conclusiones (INE 2023, reales):**
- Brecha real entre distritos: el más rico casi duplica (o más) la renta del más pobre.
- Los distritos de **mayor renta** tienden a ser los **más desiguales** por dentro.
- En los distritos ricos, **menos** parte de la renta viene del salario (más de capital/otras).

**Próximos pasos (ver `docs/PLAN_VIVIENDA.md`):**
- Añadir **alquiler** y **compra** por distrito (misma receta: URL → `extract.py` → loader) y cruzarlos con la renta para el **esfuerzo real**.
- Añadir **tipo de vivienda y m²** (Censo 2021).
- Pintar el mapa del **esfuerzo de alquiler** (dónde se vive más apurado).

**Limitaciones:** renta por hogar/persona (registros administrativos); el dato es por distrito, no por individuo.
